#### 1. Load the Flight Delay dataset exploiting Pandas APIs.

In [ ]:
# libraries
import pandas as pd
from IPython.display import display
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
DF = pd.read_csv('../../Dataset/LAB2/831394006_T_ONTIME.csv', sep=',')
display(DF)

### context of the DS
This dataset is made available by the **Bureau of Transportation Statistics** of the **United States Department of Transportation**.  
Measuring the performance of flight carriers (e.g. *American Airlines*, *EasyJet*) is extremely important for the transportation department and, for this reason, all the information related to each flight are constantly monitored and collected in huge databases by the Department of Transportation. To the aim of this laboratory, just a small set of information has been extracted.

The dataset contains the **Carrier On-Time Performance** information collected from **01-01-2017** until **31-01-2017** for all the flights in the United States. Each row represents a flight in a specific day.

Some of the most useful fields in the dataset are:

- **FL_DATE**: day of the flight in format `YYYY-mm-dd`.  
- **TAIL_NUM**: aircraft registration number, unique to a single aircraft.  
- **UNIQUE_CARRIER**: flight carrier id. --> carrier means an airline company (AA → American Airlines)
- **FL_NUM**: number of the flight.  
- **ORIGIN**: departure airport code.  
- **DEST**: destination airport code.  
- **CRS_DEP_TIME**: scheduled departure time (local time: `HHMM`) shown in the carriers’ Computerized Reservations Systems (CRS).  
- **DEP_TIME**: actual departure time (local time: `HHMM`).  
- **DEP_DELAY**: overall delay at departure. Difference in minutes (floating point number) between scheduled and actual departure time. Early departures set to 0.  
- **CRS_ARR_TIME**: scheduled arrival time (local time: `HHMM`) shown in the carriers’ Computerized Reservations Systems (CRS).  
- **ARR_TIME**: actual arrival time (local time: `HHMM`).  
- **ARR_DELAY**: overall delay. Difference in minutes (floating point number) between scheduled and actual arrival time. Early arrivals show negative numbers.  
- **CARRIER_DELAY**: delay in minutes (floating point number) caused by the carrier.  
- **WEATHER_DELAY**: delay in minutes (floating point number) caused by the weather.  
- **NAS_DELAY**: delay in minutes (floating point number) caused by the National Air System (NAS).  
- **SECURITY_DELAY**: delay in minutes (floating point number) caused by the security.  
- **LATE_AIRCRAFT_DELAY**: delay in minutes (floating point number) caused by the aircraft.  

There are some other fields in this dataset. You can explore them, understand what they represent, and whether they are significant or not for your analysis. Also, you can navigate the web page where the data has been collected from.

#### Use the info() and describe() methods to analyze how your records are distributed

> df.info() → structural summary
- shows column names, data types (int64, float64, object…),
- how many non-null (non-missing) values per column,
- total rows and memory usage.

> Example
% RangeIndex: 100 entries, 0 to 99
% Data columns (total 3):
% #  Column  Non-Null Count  Dtype
% --- -------  --------------  -----
% 0  age      100 non-null    int64
% 1  height    98 non-null    float64
% 2  name     100 non-null    object


> df.describe() → statistical summary
- works only on numeric columns by default,
- gives count, mean, std, min, quartiles, max.

> Example:
%           age     height
% count  100.00      98.00
% mean    34.21     172.45
% std     11.55      12.33
% min     18.00     145.00
% 25%     25.00     163.00
% 50%     32.00     170.00
% 75%     42.00     180.00
% max     68.00     200.00



In [ ]:
display(DF.info())
# “Non-null count” shows how many non-missing (non-NaN)

print('+++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++')

display(DF.describe())
# count, counts non-Nan values

#### which type does each column have?
you your eyes and look at the Dtype column generated by DF.info()

#### are there any missing values?
- 3   TAIL_NUM               449378 non-null  object 
- 16  DEP_TIME               441476 non-null  float64
- 17  DEP_DELAY              441476 non-null  float64
- 18  TAXI_OUT               441244 non-null  float64
- 19  WHEELS_OFF             441244 non-null  float64
- 20  WHEELS_ON              440746 non-null  float64
- 21  TAXI_IN                440746 non-null  float64
- 23  ARR_TIME               440746 non-null  float64
- 24  ARR_DELAY              439645 non-null  float64
- 26  CANCELLATION_CODE      8886 non-null    object 
- 27  CARRIER_DELAY          97699 non-null   float64
- 28  WEATHER_DELAY          97699 non-null   float64
- 29  NAS_DELAY              97699 non-null   float64
- 30  SECURITY_DELAY         97699 non-null   float64
- 31  LATE_AIRCRAFT_DELAY    97699 non-null   float64
- 32  Unnamed: 32            0 non-null       float64

All these columns has missing values!

#### how many unique carriers are present?

In [ ]:
unique_carrier = DF['UNIQUE_CARRIER'].value_counts()
display(unique_carrier)   # sorted

# OR

grouped = DF.groupby('UNIQUE_CARRIER')
display(grouped.size())     # not sorted

#### how many unique airports are present?
There's like a gigazillion airport columns, but I'll just use 'ORIGIN_CITY_NAME' and 'DEST_CITY_NAME'
There's so much almost redundant info, columns all have slightly different data

In [ ]:
origin_airport = DF['ORIGIN_CITY_NAME'].unique()
origin_airport = list(origin_airport)
arrival_airpot = DF['DEST_CITY_NAME'].unique()
arrival_airpot = list(arrival_airpot)
unique_ariport = origin_airport + arrival_airpot
unique_ariport = list(set(unique_ariport))

display(unique_ariport)
print(f'How many unique airports are in the DS? {len(unique_ariport)}')


################# I unvoluntary counted the number of flights between each pair of cities :)

grouped = DF.groupby(['ORIGIN_CITY_NAME', 'DEST_CITY_NAME'])

count = grouped.size()
count = count.reset_index(name='count_flight')
#display(count)

#### from which time interval data were collected?

In [ ]:
start = DF['FL_DATE'].min()
end = DF['FL_DATE'].max()

print(f"Data has been collected from {start} to {end}")

---

#### 3. Filter out all canceled flights.

- Look at 'CANCELLED' and 'CANCELLATION_CODE'
- If the flight hasn't been cancelled you expecte values like: 0.00,""
- If these values are different --> flight cancelled

In [ ]:
display(DF['CANCELLED'].value_counts())
# has value 1.0 if the flight was cancelled --> 8886 flights has been cancelled

display(DF['CANCELLATION_CODE'].value_counts())
# B    5327
# A    2105
# C    1167
# D     287

# let's check that the cancellation_code assumes a value only if cancelled is 1
display(DF[DF['CANCELLED'] == 1])
# yep


# cleaned_DF = DF['CANCELLED'].dropna() --> nope: which selects just one column ('CANCELLED') and removes its missing values. The result is a Series, not the full DataFrame.
# let's copy the dataframe DF, BUT let's drop the Nan values in CANCELLED column
before_drop = len(DF)
cleaned_DF = DF.dropna(subset=['CANCELLED'])
after_drop = len(cleaned_DF)
print(f"Before drop: {before_drop} rows, after drop: {after_drop} rows")
# didn't drop shit because there are no Nan values in cancelled :)
# I could have avoided it paying attention to DF.info()
#  25  CANCELLED              450017 non-null  float64
# dumbass

# keep only rows where cleaned_DF['CANCELLED'] is 0
# cleaned_DF = cleaned_DF[cleaned_DF['CANCELLED'] == 0]
cleaned_DF_1 = cleaned_DF[cleaned_DF['CANCELLED'] == 0]
display(cleaned_DF_1)

# OR
# I could have kept only those rows where 'CANCELLATION_CODE' is Nan
# 26  CANCELLATION_CODE      8886 non-null    object 
cleaned_DF_2 = cleaned_DF[cleaned_DF['CANCELLATION_CODE'].isna()]
display(cleaned_DF_2)

# same results
clean_DF = cleaned_DF_1

---

#### 4. Use any pandas method and functionality to answer the following queries:
- how many flights had each carrier operated?
- for each carrier, compute the mean delay considering all possible reasons (due to the carrier, weather, etc.)

All delay types:
- DEP_DELAY
- ARR_DELAY
- CARRIER_DELAY
- WEATHER_DELAY
- NAS_DELAY
- SECURITY_DELAY
- LATE_AIRCRAFT_DELAY

In [ ]:
# how many flights had each carrier operated?

grouped = clean_DF.groupby('UNIQUE_CARRIER')
count = grouped.size()
display(count)

In [ ]:
# for each carrier, compute the mean delay considering all possible reasons (due to the carrier, weather, etc.)

grouped = clean_DF.groupby('UNIQUE_CARRIER')
count_carrier = grouped.size() 
#display(count_carrier)
#print(type(count_carrier))


#count_carrier['mean_delay'] = 
for U_C, values in grouped:
    #print(f"\nUnique carrier: {U_C}")
    #display(values)

    tot_delays_series = values['ARR_DELAY'] + values['CARRIER_DELAY'] + values['WEATHER_DELAY'] + values['NAS_DELAY'] + values['SECURITY_DELAY'] + values['LATE_AIRCRAFT_DELAY']
    #print(f"\n##############series of delays: {tot_delays_series}")
    # each value is a series, so tot-delay will be a sieries where each row is the sum of the delays
    # I want just one value --> sum them
    # perform the sum on a series
    tot_delays = tot_delays_series.sum()
    print(f'\n+++++++++++++Total delays for {U_C}: {tot_delays}')
    mean_delay = tot_delays / len(values)
    print(f'\n-------------Mean delay for {U_C}: {mean_delay}')
# in this way there's ONE mean value per unique carrier,

# I could have also done:
for U_C, values in grouped:
    #print(f"\nUnique carrier: {U_C}")
    #display(values)

    tot_delays_series = values['ARR_DELAY'] + values['CARRIER_DELAY'] + values['WEATHER_DELAY'] + values['NAS_DELAY'] + values['SECURITY_DELAY'] + values['LATE_AIRCRAFT_DELAY']
    print(f"\n##############\nSeries of delays: {tot_delays_series}")
    
    mean_delay_series = tot_delays_series.mean()
    print(f'\n-------------\nSeries of mean delay for {U_C}: {mean_delay_series}')


# BUT I have different values ..... why?

### Handling NaN Values in Delay Calculations
When computing the mean total delay per carrier, both of the following approaches seem equivalent:

```python
# Version 1
tot_delays = tot_delays_series.sum()
mean_delay = tot_delays / len(values)
```
```python
# Version 2
mean_delay = tot_delays_series.mean()
```

However, the two methods can yield **different results** if any of the delay columns contain missing values (`NaN`).

The first version (*sum then divide*) performs a total sum over the series, but when two or more delay columns are added together, any row containing a `NaN` in any of those columns becomes `NaN` in the result.  
The final `.sum()` then ignores those missing rows, but the division by `len(values)` still uses the total number of flights, including those with incomplete data.  
This leads to an **underestimation** of the mean delay.

The second version (`.mean()`) automatically ignores `NaN` values and divides only by the number of valid rows, returning a **correct average** for available data.

If all rows are cleaned beforehand — i.e., all delay columns have valid (non-`NaN`) values — both versions will produce the **same result**.


**BUT, it's quicker to use the second method**

---

#### 4. Use any pandas method and functionality to answer the following queries:
- how many flights had each carrier operated?
- for each carrier, compute the mean delay considering all possible reasons (due to the carrier, weather, etc.)

In [ ]:
# how many flights had each carrier operated?
grouped = clean_DF.groupby('UNIQUE_CARRIER')
count_carrier = grouped.size()
display(count_carrier)

# for each carrier, compute the mean delay considering all possible reasons (due to the carrier, weather, etc.)
for group, values in grouped:
    #print(f"Unique carrier: {group}")
    #display(values)
    tot_delay = values['ARR_DELAY'] + values['CARRIER_DELAY'] + values['WEATHER_DELAY'] + values['NAS_DELAY'] + values['SECURITY_DELAY'] + values['LATE_AIRCRAFT_DELAY']
    # display(tot_delay)
    mean_delay_per_carrier = tot_delay.mean()
    print(f"Mean delay for {group}: {mean_delay_per_carrier}")



---

#### 5. Add two new columns to your DataFrame:
- weekday: it is the day of the week expressed as an integer number. Check out Pandas **dayofweek** attribute.
- delaydelta: it is the difference between the arrival delay and the departure one.

##### dayofweek
- Monday → 0
- Tuesday → 1
- …
- Sunday → 6

```python
import pandas as pd

df = pd.DataFrame({
    'date': pd.to_datetime(['2025-01-01', '2025-01-02', '2025-01-05'])
})

df['day_of_week'] = df['date'].dt.dayofweek
print(df)
```
```python
        date  day_of_week
0 2025-01-01            2   # Wednesday
1 2025-01-02            3   # Thursday
2 2025-01-05            6   # Sunday
```

In [ ]:
# clean_DF['weekday'] = clean_DF['FL_DATE'].dt.dayofweek
# gives error, as the series in the column FL_DATE is not a datetime object
# !! convert it to a datetime object !!

clean_DF = clean_DF.copy() #otherwise it gives a weird warning, because of all the previous cells that use clean_DF, so it gets confused --> just do a copy and you're Gucci
clean_DF['FL_DATE'] = pd.to_datetime(clean_DF['FL_DATE'])
clean_DF['weekday'] = clean_DF['FL_DATE'].dt.dayofweek
# display(clean_DF)

clean_DF['delaydelta'] = clean_DF['ARR_DELAY'] - clean_DF['DEP_DELAY']
display(clean_DF)

---

In [ ]:
# just noticed a dumb column, let's remove it

#print(clean_DF['Unnamed: 32'])
# print(clean_DF['Unnamed: 32'].value_counts())
# it has got only Nan values, drop it
# clean_DF = clean_DF.drop(columns=['Unnamed: 32'])
# display(clean_DF)

#### 6. Choose one of the visualization tools that you know and inspect the arrival delay as a function of the day of the week. Can you find any correlation?

This one thougher:
- you have arrival delay
- you have the day of the week

> "Inspect the arrival delay as a function of the day of the week"
- x axis: day of the week
- y axis: ARR_DELAY, like the avg arrival delay per each weekday

Plot this

In [ ]:

# practice
# just for practice, plot the number of flights per weekday

# weekdays = clean_DF['weekday'].value_counts()
# weekdays.reset_index()
# don't know why but this doesn't work, but this does
weekdays = clean_DF['weekday'].value_counts().reset_index()
weekdays.columns = ['weekday', 'count']
# display(weekdays)

fig, ax = plt.subplots(figsize=(5,5))
ax.bar(weekdays.index, weekdays['count'])
plt.show()


# practice: plot the arr_delay per unique carrier
x = clean_DF['UNIQUE_CARRIER'].unique()
y = []

grouped = clean_DF.groupby('UNIQUE_CARRIER')
for u_C, values in grouped:
    tot_arr_delay_per_carrier = values['ARR_DELAY'].sum()
    y.append(tot_arr_delay_per_carrier)

fig, ax = plt.subplots(figsize=(5,5))
ax.bar(x, y)
plt.show()






In [ ]:
x = clean_DF['weekday'].unique()
x.sort() #necessary for the plot, as the correspodning y values are sorted!

y = []
grouped = clean_DF.groupby('weekday')
for weekday, value in grouped:
    # print(f"weekday: {weekday} \nValues:")
    mean_arr_delay = value['ARR_DELAY'].mean()
    # print(f"weekday: {weekday} \nMean arr delay: {mean_arr_delay}")
    y.append(float(mean_arr_delay))

# print(x)
# print(y)

fig, ax = plt.subplots(figsize=(5,5))
ax.bar(x,y)

ax.set_xticks(x) # defines where the labels appear (the tick positions).
ax.set_xticklabels(['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']) #defines what text appears at those tick positions.
ax.set_ylabel('Mean arrival delay')
plt.show




---

#### 7. Consider the weekend days only, compute, for each carrier, the mean arrival delay.

#### Now consider the working days and compute, for each carrier, the mean arrival delay.

In [ ]:
# Consider the weekend days only, compute, for each carrier, the mean arrival delay.

# grouped = clean_DF.groupby(['UNIQUE_CARRIER', 'weekday'])
# for (U_C, w_d), values in grouped:
#     if w_d == 5 or w_d == 6:
#         # print(f"Unique carrier: {U_C}")
#         # print(f"Weekday: {w_d}")
#         # display(values)
#         arr_delay_per_carrier = values['ARR_DELAY'].mean()
#         print(f"\nthe unique carrier {U_C} \non the day {w_d}\nhas an average arrival delay of {arr_delay_per_carrier}")







###########################################################################################################################################################################

# write it easier, no manual for loops
mask = clean_DF['weekday'].isin([5, 6])   # Boolean Series → True for rows where weekday = 5 or 6 --> like saying keep only rows where weekday = 5 or 6
filtered = clean_DF[mask]                 # apply the mask
weekend_DF = filtered.copy()                 # just to avoid Pandas error: “SettingWithCopyWarning”
# display(weekend_DF)

grouped = weekend_DF.groupby(['UNIQUE_CARRIER', 'weekday'])

# FUUUUUUUCK, I JUST REALISED THAT EVERYTIME I GROUP I CYCLE INSIDE THE GORUP, BUT I DON'T NEED TO, I CAN JUST DO THIS:
mean_delay = grouped['ARR_DELAY'].mean().reset_index()
# for each group, which have same UNIQUE_CARRIER (AA) and weekday (5),
# takes the series in the column ARR_DELAY and computes the mean --> one value
# I expect a mean value for UNIQUE_CARRIER (AA) and weekday (5), then for the same UNIQUE_CARRIER (AA) but the following weekend day (6)
# display(mean_delay)

mean_delay.rename(columns={'ARR_DELAY': 'avg_arr_delay'}, inplace=True)
display(mean_delay)

# No idea why would I pivot
pivot = mean_delay.pivot(
    index='weekday',
    columns='UNIQUE_CARRIER',
    values='avg_arr_delay'
)
display(pivot)
# guess it's more compact
# try to plot it and you'll see:
# when you call .plot
# 	•	Index → x-axis
#	•	Each column → one set of bars (or one line, if line plot)
#	•	Cell values → bar heights (y-axis)
#
# the x values goes from 0 to 23 --> which are the unique_carriers, then you plot weekdays (makes no sense) and the average delay
# that's why we pivot

# plot
# I want to plot:
# 1, weekend days
# 2, mean arrival delay
# 3, for each unique carrier

# I could have a bar chart, on the x 5 and 6, on the y the mean arrival delay and have 2 bar chart per unique carrier,
# each unique carrier has its own color and, for example, the bar chart for the unique carrier AA on the day 5 is next
# to the bar chart of the unique carrier AS on day 5
# I must have a legenda for showing the association color - unique carrier

fig, ax = plt.subplots(figsize=(10, 5))
#mean_delay.plot(
#    kind='bar',
#    ax=ax
#)

pivot.plot(
    kind='bar',
    ax=ax
)

ax.set_xlabel('Weekday')
ax.set_ylabel('Mean arrival delay (minutes)')
ax.set_title('Mean arrival delay on weekend days by carrier')
ax.set_xticklabels(['Saturday', 'Sunday'], rotation=0)
ax.legend(title='Carrier')

plt.show()

In [ ]:
# Now consider the working days and compute, for each carrier, the mean arrival delay.

mask = clean_DF['weekday'].isin([0, 1, 2, 3, 4])  
clean_DF = clean_DF[mask]
# display(clean_DF)

grouped = clean_DF.groupby(['UNIQUE_CARRIER', 'weekday'])
# just to visualise the groupby but not necessary:
# for (unique_carrier, weekday), values in grouped:
#     print(f"Unique carrier: {unique_carrier}")
#     print(f"Weekeday: {weekday}")
#     display(values)
mean_arr_delay_per_carrier = grouped['ARR_DELAY'].mean().reset_index()
display(mean_arr_delay_per_carrier)

pivot = mean_arr_delay_per_carrier.pivot(
    index= 'weekday',
    columns= 'UNIQUE_CARRIER',
    values= 'ARR_DELAY'
)
display(pivot)

fig, ax = plt.subplots(figsize=(10,10))
pivot.plot(
    kind='bar',
    ax=ax
)

ax.set_xlabel('Weekday')
ax.set_ylabel('Mean arrival delay (minutes)')
ax.set_title('Mean arrival delay on working days by carrier')
ax.set_xticklabels(['Monday', 'Tuesday', 'Wednesday', 'Thrusday', 'Friday'], rotation=0)
ax.legend(title='Carrier')

plt.show()



#### Then, compare the delays in working days and in weekends for each company.

In [ ]:
# ignore
cleaned_DF = DF.dropna(subset=['CANCELLED'])
cleaned_DF = cleaned_DF[cleaned_DF['CANCELLED'] == 0]
clean_DF['FL_DATE'] = pd.to_datetime(clean_DF['FL_DATE'])
clean_DF['weekday'] = clean_DF['FL_DATE'].dt.dayofweek
clean_DF = clean_DF.copy()

grouped = clean_DF.groupby(['UNIQUE_CARRIER', 'weekday'])
# for (unique_carrier, weekday), values in grouped:
#     print(f"Unique carrier: {unique_carrier}")
#     print(f"Weekday: {weekday}")
#     display(values)

mean_delay = grouped['ARR_DELAY'].mean().reset_index()
mean_delay.rename(columns={'ARR_DELAY': 'avg_arr_delay'}, inplace=True)
# display(mean_delay)

pivot = mean_delay.pivot(
    index='weekday',
    columns='UNIQUE_CARRIER',
    values='avg_arr_delay'
)
display(pivot)

fig, ax = plt.subplots(figsize=(15,8))
pivot.plot(
    kind='bar',
    ax=ax
)

ax.set_xlabel('Weekday')
ax.set_ylabel('Mean arrival delay (minutes)')
ax.set_title('Mean arrival delay on a week by carrier')
ax.set_xticklabels(['Monday', 'Tuesday', 'Wednesday', 'Thrusday', 'Friday', 'Saturday', 'Sunday'], rotation=0)
ax.legend(title='Carrier')

plt.show()

---

###### 8. CreateaPandasDataFramewithamulti-indexcomposedofthecolumns: tunique_carrier, origin, dest, fl_date.
**Hell nah**

#### 9. For each flight operated by American Airlines (AA) and Delta Airlines (DL), taken off (che parte da) from the Los Angeles International Airport (LAX) (ORIGIN == 'LAX') and for each date, display the departure time and delay.

- mask to just keep UNIQUE_CARRIER AA and DL
- mask ORIGIN == 'LAX'
- group by UNIQUE_CARRIER and FL_DATE
- show flight info + DEP_TIME and DEP_DELAY

In [ ]:
mask_1 = clean_DF['UNIQUE_CARRIER'].isin(['AA', 'DL'])
masked_DF_1 = clean_DF[mask_1]
mask_2 = masked_DF_1['ORIGIN'] == 'LAX'
masked_DF_2 = masked_DF_1[mask_2]


grouped = masked_DF_2.groupby(['UNIQUE_CARRIER', 'FL_DATE'])
for (unique_carrier, fl_date), values in grouped:
    print(f"For the date: {fl_date}\nthe unique carrier: {unique_carrier} \nThe flight depratures are:")
    display(values['DEP_TIME'])
    print("\nThe departure delays are:")
    display(values['DEP_DELAY'])

---

#### 10. For each flight that flew in the first week of the month, with LAX as destination airport, compute the mean arrival delay.
- masking: FL_DATE < 7 (I don't know if you can do it with dates, they have a weird format and weird properties)
- masking: DEST == 'LAX'
- for these flights with FL_DATE < 7 and DEST == 'LAX', compute just one value --> ARR_DELAY.mean()

In [4]:
#ignore
DF = pd.read_csv('../../Dataset/LAB2/831394006_T_ONTIME.csv', sep=',')
cleaned_DF = DF.dropna(subset=['CANCELLED'])
cleaned_DF = cleaned_DF[cleaned_DF['CANCELLED'] == 0]
clean_DF = cleaned_DF
clean_DF['FL_DATE'] = pd.to_datetime(clean_DF['FL_DATE'])
clean_DF['weekday'] = clean_DF['FL_DATE'].dt.dayofweek
clean_DF['delaydelta'] = clean_DF['ARR_DELAY'] - clean_DF['DEP_DELAY']

clean_DF.drop(columns=['Unnamed: 32'], inplace=True)
clean_DF = clean_DF.copy()

In [ ]:
mask_1 = clean_DF['FL_DATE'] <= '2017-01-07'
clean_DF_mask_1 = clean_DF[mask_1]

mask_2 = clean_DF['DEST'] == 'LAX'
clean_DF_mask_2 = clean_DF_mask_1[mask_2]

mean_arr_delay = clean_DF_mask_2['ARR_DELAY'].mean()
print(f"the mean arrival delay for dest LAX on flight from 2017-01-01 to 2017-01-07 is: {mean_arr_delay}")

---

#### Generate a pivot table containing the number of departed flights for each carrier and for each day of the week and show it.

> PIVOT TABLE ??
- A pivot table in pandas is a compact summary table created with DataFrame.pivot_table().
- It lets you aggregate data (e.g. counts, means, sums) across two dimensions — one becomes the rows (index), one the columns.

```python
pivot = clean_DF.pivot_table(
    values='',          # any column, used only to count rows
    index='',           # rows → each carrier
    columns='',         # columns → day of the week
    aggfunc=''          # count how many flights
)
display(pivot)
```

#### Generate a pivot table containing the number of departed flights for each carrier and for each day of the week and show it.
- 'number of departed flights for each carrier and for each day of the week'
* number of departed flights = values --> count the flight per carrier per day of the week
* for each carrier = UNIQUE_CARRIER on the index (or columns, up to you)
* for each day of the week = weekday on the columns (or index)

In [ ]:
grouped = clean_DF.groupby(['UNIQUE_CARRIER', 'weekday'])
count = grouped.size().reset_index(name='count_flight')
#display(count)

# pivot = clean_DF.pivot_table(
#     index = count['UNIQUE_CARRIER'],
#     columns = count['weekday'],
#     values = count['count_flight']
# )
# SECOND TIME YOU MADE THIS MISTAKE --> YOU WROTE **clean_DF**.pivot_table,
# SO INSIDE THE PIVOT TABLE JUST SPECIFY WHAT COLUMN/INDEX/VALUE YOU WANT TO USE **FROM clean_DF** --> NOT clean_df['UNIQUE_CARRIER'], just 'UNIQUE_CARRIER'

pivot_1 = count.pivot_table(
    index = 'UNIQUE_CARRIER',
    columns = 'weekday',
    values = 'count_flight'
)
display(pivot_1)

# OR to make it quicker use aggfunc inside the pivot table
pivot_2 = clean_DF.pivot_table(
    values='FL_DATE',        # pivot_table() needs some column to aggregate — it doesn’t matter which one as long as it’s not all NaN --> count how many non-null rows exist per (carrier, weekday).” So values='FL_DATE' just gives Pandas something to count.
    index='UNIQUE_CARRIER',  
    columns='weekday',       
    aggfunc='count'          
)
display(pivot_2)

# they're the same

pivot_2 = clean_DF.pivot_table(
    values='FL_DATE',        # pivot_table() needs some column to aggregate — it doesn’t matter which one as long as it’s not all NaN --> count how many non-null rows exist per (carrier, weekday).” So values='FL_DATE' just gives Pandas something to count.
    index='weekday',  
    columns='UNIQUE_CARRIER',       
    aggfunc='count'          
)
display(pivot_2)
# shows the number of flights per carrier and per day

#### Compute now the pairwise correlation between the carriers and show it on a heatmap.
- use .corr()
- consider AA, AS and show the correlation matrix between thoe two
- and then show it on a heatmap:

> what's a heatmap?

In [ ]:
# example
plt.figure(figsize=(10,6))
sns.heatmap(pivot_2, cmap='coolwarm', annot=True, fmt='d')
plt.title("Flights per Carrier and Weekday")
plt.xlabel("Weekday")
plt.ylabel("Carrier")
plt.show()

In [ ]:
# Compute now the pairwise correlation between the carriers and show it on a heatmap.

# Compute pairwise correlation between carriers
carrier_correlation = pd.DataFrame(pivot_2.corr())
# IMPORTANT: COMPUTES CORRELATION ON THE COLUMNS --> **TAKES COLUMNS AND PASTES THEM IN THE ROWS TOO**
# ** BE CAREFUL WHO IS IN THE COLUMNS, YOU'LL COMPUTE CORRELATION MATRIX FOR THEM **

# Plot the correlation matrix as a heatmap.
plt.figure(figsize=(10,6))
sns.heatmap(carrier_correlation, cmap='coolwarm', annot=True, fmt='.2f')      # fmt='d'is for integers, fmt='.2f' is for floats
plt.title("Carriers correlation")
plt.xlabel("x")
plt.ylabel("y")
plt.show()

# pivot_2 shows the number of flights per carrier and per day
# carrier_correlation shows the correlation between the carriers, so if ≈ 1.0 it means that both AA and B6 has the same level of business, like they both had many flights or they both had very few flights
# then the heatmap shows the exact same info, but it's visually more appealing


---

#### 12. Generate a pivot table containing the average arrival delay, for each carrier and for each day of the week and show it.
- group by UNIQUE_CARRIER, weekday
- avg(ARR_DELAY)

#### Compute now the pairwise correlation between the carriers and show it on a heatmap.
same as before:
- compute correlation matrix --> .corr()
- NOTE: carries on the column so that the correlation matrix is computed on the carries
- plot the heatmap

• What does this correlation matrix represent?
• Can you find any carrier with different delay behaviors?

In [ ]:
DUMMY = False
if DUMMY:
    group = clean_DF.groupby(['UNIQUE_CARRIER', 'weekday'])

    # for (carrier, weekday), values in group:
    #     print(f"Carrier: {carrier} \nWeekday: {weekday} \n")
    #     display(values)

    pivot = group.pivot_table(
        values= 'ARR_DELAY',
        index='weekday',  
        columns='UNIQUE_CARRIER',
        aggfunc='mean'
    )
    display(pivot)

# AttributeError: 'DataFrameGroupBy' object has no attribute 'pivot_table'
# YOU CANNOT PERFORM A PIVOT TABLE ON A GROUPED DATAFRAME (when you group a DF it's not a DF anymore it's a weird object: 'DataFrameGroupBy')
# AND YOU CANNOT PERFORM PIVOT TABLE ON THOSE

# BUT YOU DON'T EVEN NEED TO GROUP BY, THE PIVOT TABLE ALREADY GROUPS BY, SO DIRECTLY WORK ON clean_DF !!!!!!!!!!

pivot = clean_DF.pivot_table(
    values= 'ARR_DELAY',
    index = 'weekday',
    columns = 'UNIQUE_CARRIER',
    aggfunc = 'mean'
)
display(pivot)

# compute correlation matrix --> .corr()
correlation_matrix = pivot.corr()
display(correlation_matrix)

# plot the same matrix but with a heatmap
plt.figure(figsize=(10,6))
sns.heatmap(correlation_matrix, cmap='coolwarm', annot=True, fmt='.2f')      # fmt='d'is for integers, fmt='.2f' is for floats
plt.title("Carriers correlation")
plt.xlabel("x")
plt.ylabel("y")
plt.show()



---

#### 13. Using a pivot table, for the carriers HA, DL, AA and AS compute the average deltadelay for each day of the week.
- mask clean_DF --> 'UNIQUE_CARRIER' in ['HA', 'DL', 'AA', 'AS']
- groupby weekday and these UNIQUE_CARRIER --> not necessary, the pivot_table already does this for you!

#### Then, display the results on a line plot, having a line per carrier and the weekday on the x-axis.
line plot ???
ooohhhh it makes sense actually:
- for the weekday 0 you'll have 3 values (AA, AS, DL, HA) per row
- draw a line passing trough these 3 points and do it for each row



In [ ]:
# mask = clean_DF['UNIQUE_CARRIER'] in ['HA', 'DL', 'AA', 'AS'] ** 'in' doesn't work on series **

mask = clean_DF['UNIQUE_CARRIER'].isin(['HA', 'DL', 'AA', 'AS'])
masked_DF = clean_DF[mask]

pivot = masked_DF.pivot_table(
    values = 'delaydelta',
    index= 'weekday',               # index == x axis
    columns = 'UNIQUE_CARRIER',     # columns == y axis
    aggfunc = 'mean'
)
display(pivot)

# plot line plot
pivot.plot(kind='line', marker='o', figsize=(8,5))
plt.title("Mean delay per carrier across weekdays")
plt.xlabel("Weekday (0=Mon, 6=Sun)")
plt.ylabel("Mean delay (minutes)")
plt.legend(title='Carrier')
plt.show()